# 01 — Train Retinexformer + selective gate

**Run `00_kaggle_setup.ipynb` first** (this notebook re-creates the environment
in its first cell so it can also run standalone).

This kernel trains **`configs/baseline.yaml`** (budget: 50000 iters).

1. Start with `configs/smoke.yaml` (~1 min) to verify the whole pipeline.
2. Then switch to `configs/gate_pixel.yaml` (the real run: `train.iters: 150000`,
   ≈2–4 h on a Kaggle T4 — check the session timer; use
   `--set train.iters=50000` if you are tight on time).

Checkpoints and logs land in `/kaggle/working/runs/<name>/` — they are saved as
**notebook output** and can be downloaded from the Output tab (or
`kaggle kernels output <slug>`).

## 1. Environment (clone + install — same as 00, no secrets needed)

In [ ]:
REPO_URL = "https://github.com/pranjalkole/selective-gating.git"   # public repo
BRANCH   = "master"

import os, subprocess, sys, shutil
from pathlib import Path

repo = Path("/kaggle/working/selective-gating")
if repo.exists():
    if subprocess.run(["git", "-C", str(repo), "pull", REPO_URL, BRANCH]).returncode != 0:
        print("pull failed -> fresh clone")
        shutil.rmtree(repo)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)

print("repo at:", repo)

src = repo / "src"
sys.path.insert(0, str(src))   # PYTHONPATH only affects child processes, not this kernel
os.environ["PYTHONPATH"] = f"{src}:{os.environ.get('PYTHONPATH', '')}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)

import selgate
print("selgate", selgate.__version__)

## 2. Config + run training (`configs/baseline.yaml`)

Override `CONFIG` for other runs, or use the template `01_train.ipynb`.

Optional extras:
- warm-start / frozen-backbone ablation (weights dataset is auto-attached):
  `OVERRIDES += ["--set", "model.backbone.pretrained=/kaggle/input/retinexformer-weights/LOL_v1.pth"]`

In [ ]:
CONFIG   = 'configs/baseline.yaml'
OVERRIDES = ["--set", "train.iters=50000"]

cmd = [sys.executable, str(repo / "scripts" / "train.py"), "--config", str(repo / CONFIG), *OVERRIDES]
# auto-resume: if a previous session was killed at the timeout, last.pt is in
# /kaggle/working (persisted notebook output) and training continues from it
from selgate.config import load_config
RUN_NAME = load_config(repo / CONFIG).get_path("name")   # yaml name, not the file stem
_last = Path("/kaggle/working/runs") / RUN_NAME / "last.pt"
if _last.exists():
    cmd += ["--resume", str(_last)]
    print("resuming from", _last)
print(" ".join(cmd))
ret = subprocess.run(cmd, cwd=repo).returncode
assert ret == 0, f"training failed (exit {ret})"

## 3. Training curves + gate monitor

In [ ]:
import json
from pathlib import Path
from selgate.config import load_config

name = load_config(repo / CONFIG).get_path("name")   # yaml name, not the file stem
run_dir = Path("/kaggle/working/runs") / name
hist_path = run_dir / "history.json"
if not hist_path.exists():
    # train.out_dir may be overridden by SELGATE_OUT_DIR
    candidates = list(Path("/kaggle/working").rglob("history.json"))
    hist_path = candidates[-1] if candidates else None
assert hist_path and hist_path.exists(), "history.json not found under /kaggle/working"

hist = json.loads(hist_path.read_text())
iters = [h["iter"] for h in hist]

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(iters, [h["total"] for h in hist], label="total")
axes[0].plot(iters, [h["l1"] for h in hist], label="l1")
axes[0].plot(iters, [h.get("illum", 0) for h in hist], label="illum")
axes[0].set_title("losses"); axes[0].legend(); axes[0].set_xlabel("iter")
if "gate_mean" in hist[0]:
    axes[1].plot(iters, [h["gate_mean"] for h in hist], label="gate mean")
    axes[1].fill_between(iters,
                         [h["gate_mean"] - h["gate_std"] for h in hist],
                         [h["gate_mean"] + h["gate_std"] for h in hist], alpha=0.3)
    axes[1].set_title("gate mean ± std"); axes[1].set_xlabel("iter")
    axes[2].plot(iters, [h["frac_open"] for h in hist], label="frac open (g>0.9)")
    axes[2].plot(iters, [h["frac_closed"] for h in hist], label="frac closed (g<0.1)")
    axes[2].set_title("collapse monitor"); axes[2].legend(); axes[2].set_xlabel("iter")
plt.tight_layout(); plt.show()

print("checkpoints:", sorted(p.name for p in run_dir.glob('*.pt')))
print("val PSNR history: see stdout above (val @ ... lines)")

## 4. Grab the checkpoint

Download `best.pt` from the **Output** pane on the right
(file tree: `runs/<name>/best.pt`), or from a local terminal:

```bash
kaggle kernels output <your-slug>/<notebook-name> -p ./downloaded
```

Then run `02_evaluate.ipynb` (either in the same Kaggle session or a new one
with the checkpoint attached as a dataset).